### Imports

In [25]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as dataset
import deep_learning_land_use_classification.evaluation as evaluation

# External imports
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="sstaheli52-wageningen-university-and-research",
    
    # Set the wandb project where this run will be logged.
    project="DL_land-use-classification",
    dir = dataset.WANDB_DIR,

    # Track hyperparameters and run metadata.
    config={
        "architecture": "resnet50",
        "pretrained": True,
        "pretraining_dataset": "ImageNetV2",
        "pretraining_source": "torchvision",
        "weights": "IMAGENET1K_V2",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "num_classes": dataset.num_classes
        },
)

config = run.config

### Resize, transform and normalize dataset

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # standard ImageNet mean
        std =[0.229, 0.224, 0.225] # standard ImageNet std
    )
])

### Get training and test dataset, as well as dataset loaders

In [ ]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return image, label

In [ ]:
train_dataset = MultiLabelDataset(dataset.train_df, dataset.class_names, transform)
test_dataset  = MultiLabelDataset(dataset.test_df, dataset.class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

### Initiate model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, dataset.num_classes)
model = model.to(device)

Using device: cuda


We use BCEWithLogitsLoss because our goal is multiclass classification. We also penalize larger classes using pos_weight due to the class imbalance present in the dataset. 

In [ ]:
labels = dataset.train_df[dataset.class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

In [ ]:
run.watch(model, log="all", log_freq=100)

### Train model

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
epochs = config.epochs

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics(
        model,
        test_loader,
        device,
        threshold=0.5
    )

    # Log metrics to Weights & Biases
    run.log({"train_loss": train_loss, "test_loss": test_loss})
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(dataset.class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))

    run.log({
        "train_loss": train_loss,
        "test_loss": test_loss,
        "macro/precision": p_macro,
        "macro/recall": r_macro,
        "macro/f1": f1_macro,
        "class_metrics": class_metrics,
    })
    

Epoch 1/5
Train Loss: 0.1023
Test Loss:  0.1702
Epoch 2/5
Train Loss: 0.0837
Test Loss:  0.1762
Epoch 3/5
Train Loss: 0.0643
Test Loss:  0.1389
Epoch 4/5
Train Loss: 0.0477
Test Loss:  0.1562
Epoch 5/5
Train Loss: 0.0388
Test Loss:  0.2063


### Evaluate model performance

In [ ]:
precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics(
    model,
    test_loader,
    device,
    threshold=0.5
)

for i, cls in enumerate(dataset.class_names):
    print(f"{cls:15s} | Precision: {precision[i]:.3f} | Recall: {recall[i]:.3f} | F1: {f1[i]:.3f}")

print("\n--- Macro Averages ---")
print(f"Precision: {p_macro:.3f}")
print(f"Recall:    {r_macro:.3f}")
print(f"F1 Score:  {f1_macro:.3f}")

airplane        | Precision: 1.000 | Recall: 1.000 | F1: 1.000
bare-soil       | Precision: 0.824 | Recall: 0.771 | F1: 0.797
buildings       | Precision: 0.915 | Recall: 0.929 | F1: 0.922
cars            | Precision: 0.904 | Recall: 0.829 | F1: 0.865
chaparral       | Precision: 0.931 | Recall: 1.000 | F1: 0.964
court           | Precision: 1.000 | Recall: 0.957 | F1: 0.978
dock            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
field           | Precision: 1.000 | Recall: 1.000 | F1: 1.000
grass           | Precision: 0.882 | Recall: 0.891 | F1: 0.887
mobile-home     | Precision: 0.964 | Recall: 1.000 | F1: 0.982
pavement        | Precision: 0.931 | Recall: 0.884 | F1: 0.907
sand            | Precision: 0.869 | Recall: 0.914 | F1: 0.891
sea             | Precision: 1.000 | Recall: 0.900 | F1: 0.947
ship            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
tanks           | Precision: 0.947 | Recall: 1.000 | F1: 0.973
trees           | Precision: 0.879 | Recall: 0.875 | F1

In [ ]:
# Log evaluation metrics to Weights & Biases
run.log({
    "macro/precision": p_macro,
    "macro/recall": r_macro,
    "macro/f1": f1_macro,
})

class_metrics_table = wandb.Table(columns=["class", "precision", "recall", "f1"])
for i, cls in enumerate(dataset.class_names):
    class_metrics_table.add_data(cls, float(precision[i]), float(recall[i]), float(f1[i]))

run.log({"class_metrics": class_metrics_table})

run.finish()

macro/f1,▁
macro/precision,▁
macro/recall,▁
macro/f1,0.93898
macro/precision,0.94392
macro/recall,0.93518
